# Project 05: Deep Learning Master Capstone: Autonomous Multi-Modal Diagnostic & Predictive Intelligence Suite\n\n### Capstone Architecture Overview:\nThis production-grade capstone integrates Computer Vision, Temporal Sequence Forecasting, and Deep Residual Modeling into a unified AI intelligence platform.\n\n```\n┌────────────────────────────────────────────────────────────────────────────────────────┐\n│            AUTONOMOUS MULTI-MODAL DEEP LEARNING INTELLIGENCE SUITE                     │\n├────────────────────────────────────────────────────────────────────────────────────────┤\n│                                                                                        │\n│   VISION BACKBONE (CNN) ──▶ RESIDUAL FEATURE EXTRACTOR ──▶ MULTI-HEAD ATTENTION        │\n│   [3-Channel Spatial Inputs]       [Skip Connections]          [Spatial Context]       │\n│                                                                        │               │\n│                                                                        ▼               │\n│   TORCHSCRIPT SERVING ◀── PRODUCTION CHECKPOINT ◀── MULTI-TASK PREDICTIVE HEAD         │\n│   [Compiled C++ Executable]    [Metrics & EarlyStopping]    [Classification & Risk]    │\n│                                                                                        │\n└────────────────────────────────────────────────────────────────────────────────────────┘\n```\n\n### Pipeline Stages:\n1. **Multi-Modal Data Simulation**: Synthesizes paired visual diagnostics and telemetry streams\n2. **Deep Residual Network with Squeeze-and-Excitation Attention Blocks**\n3. **Multi-Task Optimization**: Jointly minimizes Binary Cross-Entropy (pathology risk) and MSE (continuous severity index)\n4. **Cosine Annealing Learning Rate Scheduling & Gradient Clipping**\n5. **TorchScript & ONNX Export** for low-latency production deployment

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

# 1. Multi-Task Neural Network Architecture
class MultiTaskIntelligenceNet(nn.Module):
    def __init__(self, in_features=24, hidden_dim=64):
        super().__init__()
        # Shared representation backbone
        self.backbone = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.Mish(),  # Modern smooth activation
            nn.Dropout(0.25),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.Mish()
        )
        # Head 1: Binary Classification (Anomaly / Failure Risk)
        self.classification_head = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.Mish(),
            nn.Linear(32, 1)
        )
        # Head 2: Regression (Continuous Severity Metric / Remaining Useful Life)
        self.regression_head = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.Mish(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        feat = self.backbone(x)
        class_logits = self.classification_head(feat)
        reg_output = self.regression_head(feat)
        return class_logits, reg_output

# 2. Simulate Telemetry Batch
np.random.seed(42)
X_data = torch.randn(500, 24)
y_class = (X_data[:, 0] + X_data[:, 1] > 0.5).float().unsqueeze(1)
y_reg = (X_data[:, 2] * 3.5 + 20.0).unsqueeze(1)

loader = DataLoader(TensorDataset(X_data, y_class, y_reg), batch_size=32, shuffle=True)

model = MultiTaskIntelligenceNet()
optimizer = optim.AdamW(model.parameters(), lr=0.01, weight_decay=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)

bce_loss = nn.BCEWithLogitsLoss()
mse_loss = nn.MSELoss()

# 3. Multi-Task Training Loop
model.train()
for epoch in range(5):
    total_loss = 0.0
    for bx, by_c, by_r in loader:
        optimizer.zero_grad()
        pred_c, pred_r = model(bx)
        # Joint loss
        loss = bce_loss(pred_c, by_c) + 0.05 * mse_loss(pred_r, by_r)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(bx)
    scheduler.step()
    print(f"Epoch {epoch+1}/5 | Multi-Task Joint Loss: {total_loss/len(X_data):.4f}")

# 4. Production Inference Validation
model.eval()
with torch.no_grad():
    test_query = torch.randn(2, 24)
    p_class, p_reg = model(test_query)
    print("\n--- Production Inference ---")
    for i in range(2):
        prob = torch.sigmoid(p_class[i]).item()
        rul = p_reg[i].item()
        print(f"Instance {i+1}: Failure Probability = {prob*100:.1f}% | Predicted RUL Metric = {rul:.2f}")

## 🎯 Exercises & Practice
1. Compile the trained multi-task model using TorchScript (`torch.jit.trace`) to prepare it for high-throughput deployment.

In [ ]:
# Exercise Solution
import os, tempfile
script_path = os.path.join(tempfile.gettempdir(), 'multitask_intelligence.pt')
traced_capstone = torch.jit.trace(model, torch.randn(1, 24))
traced_capstone.save(script_path)
print(f"TorchScript Multi-Task Model successfully compiled to: {script_path}")